# Thermostats in Molecular Dynamics Simulations

In molecular dynamics simulations, thermostats are employed to control the temperature of the system by adjusting particle velocities. This section provides an overview of the thermostats used in this work, their theoretical background, and a comparative analysis of their results.

In the main code of [MD simulation](), the thermostat is applied during the main integration loop by calling the relevant library. The user can choose whether to apply a thermostat and specify its type as a compile-time parameter. The primary purpose of using thermostats in this simulation is to study **phase transitions** and the annealing process. These processes are controlled using a [bash script]() that adjusts the system temperature across consecutive simulations. 

All implementations are included in the [Thermostat.h]() library, covering three of the most commonly used thermostats in molecular dynamics literature: **Velocity Scaling**, **Berendsen**, and **Andersen** thermostats, which will be explained in detail below.

---

### Outline:
1. **Thermostats Overview**
    - **Velocity Scaling Thermostat**
    - **Berendsen Thermostat**
    - **Andersen Thermostat**
2. **Comparative Analysis of Thermostat Performance**

---

## 1. Thermostats Overview

As mentioned earlier, thermostats control temperature by manipulating particle velocities based on the principle of **equipartition of energy**. According to this principle, each degree of freedom contributes equally to the system's temperature through its kinetic energy. 

For a more detailed explanation of temperature calculation and the correction of the center-of-mass velocity, refer to the [Energy and Kinetic Properties Notebook](). The fundamental idea is to update translational and rotational velocities, $ \mathbf{v} $ and $ \mathbf{\omega} $, based on the target temperature calculated using the formula:

$$
T = \frac{2}{f} \frac{K_{\text{total}}}{N k_B}
$$

where:  
- $ T $ is the temperature.  
- $ f $ is the number of degrees of freedom.  
- $ K_{\text{total}} $ is the total kinetic energy.  
- $ N $ is the number of particles.  
- $ k_B $ is the Boltzmann constant.  

The total kinetic energy is calculated as:

$$
K_{\text{total}} = \sum_{i=1}^{N} \frac{1}{2} \left( m v_i^2 + I \omega_i^2 \right)
$$

where $ m $ and $ I $ are the mass and moment of inertia of a particle, respectively. For further details, please refer to the [Energy and Kinetic Properties Notebook]().

---

### 1.1 Velocity Scaling Thermostat
The **Velocity Scaling Thermostat** maintains the system temperature by uniformly scaling the velocities of all particles at each timestep. It directly adjusts the velocities based on the ratio of the current temperature to the target temperature:

$$
\mathbf{v'} = \lambda \mathbf{v}, \quad \mathbf{\omega'} = \lambda \mathbf{\omega}
$$

where the scaling factor $ \lambda $ is given by:

$$
\lambda = \sqrt{\frac{T_{\text{target}}}{T_{\text{current}}}}
$$

- $ T_{\text{target}} $: Desired temperature.  
- $ T_{\text{current}} $: Instantaneous kinetic temperature.

Velocity scaling offers a simple and direct method for controlling temperature but lacks physical accuracy since it forces the temperature without natural energy fluctuations.

---

### 1.2 Berendsen Thermostat
The **Berendsen Thermostat** is a more refined version of velocity scaling, introducing a relaxation parameter for smoother temperature control. It scales the velocities similarly but with a gradual adjustment:

$$
\lambda = \sqrt{1 + \frac{\Delta t}{\tau} \left( \frac{T_{\text{target}}}{T_{\text{current}}} - 1 \right)}
$$

where:  
- $ \Delta t $: Timestep size.  
- $ \tau $: Relaxation time constant, controlling how quickly the temperature converges to the target value.  

The Berendsen thermostat provides smoother temperature control but does not fully sample the canonical ensemble due to continuous velocity adjustments.

---

### 1.3 Andersen Thermostat
The **Andersen Thermostat** regulates the temperature using stochastic velocity randomization, modeling the effect of a heat bath through random collisions. This method periodically resets particle velocities, drawing them from a Maxwell-Boltzmann distribution consistent with the target temperature. 

The thermostat works as follows:

#### Collision Frequency
At each timestep, a particle has a probability of undergoing a collision defined by:

$$
P = \nu \Delta t
$$

where:  
- $ \nu $ = Collision frequency parameter (controlling how often velocities are randomized).  
- $ \Delta t $ = Simulation timestep.  

#### Velocity Reassignment
When a collision occurs, the particle's velocity is redrawn from the **Maxwell-Boltzmann distribution**:

$$
f(v) = \sqrt{\frac{m}{2 \pi k_B T}} \exp{\left(-\frac{m v^2}{2 k_B T}\right)}
$$

where:  
- $ m $ = Particle mass.  
- $ k_B $ = Boltzmann constant.  
- $ T $ = Target temperature.  

This distribution ensures that the system samples the canonical ensemble accurately, making the Andersen thermostat suitable for equilibrium simulations.

**Summary:** The Andersen thermostat provides proper sampling of the canonical ensemble but introduces discontinuities in particle trajectories due to the random reassignment of velocities.

---

## 2. Comparative Analysis of Thermostat Performance

### 2.1 Temperature Control Behavior
A comparison of the temperature control behavior for the three thermostats:

- **Velocity Scaling:** Sharp adjustments, instant temperature control but unrealistic for physical simulations.  
- **Berendsen:** Smooth control, but non-canonical behavior due to continuous scaling.  
- **Andersen:** Accurate temperature sampling with fluctuations due to random velocity reassignment.

---

### 2.2 Visualizing Results
Below is an example of visualizing temperature evolution using these thermostats.